In [46]:
import logging
import math
import torch
from einops import rearrange,einsum
from cs336_basics import custom_training, custom_modules,io_utils,tokenizer
import numpy as np

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
# Create a console handler
console_handler = logging.StreamHandler()
# Create a formatting layout
formatter = logging.Formatter('%(name)s - %(levelname)s - %(asctime)s - %(message)s')
console_handler.setFormatter(formatter)

# Add the handler to your logger
logger.addHandler(console_handler)

In [ ]:
vocab = io_utils.deserialize_vocab("../data/vocab_tinystories_sample_5M.txt")
merges = io_utils.deserialize_merges("../data/merges_tinystories_sample_5M.txt")
special_tokens=['<|endoftext|>']
tkn = tokenizer.Tokenizer(vocab = vocab, merges = merges, special_tokens=special_tokens)

prompt = "He saw"

context_length = 256
device = "cpu"
token_ids = tkn.encode(prompt)
prompt_len = len(token_ids)
# token_ids
input = torch.tensor( token_ids,dtype=torch.long).view(1, prompt_len).to(device)
logger.debug(f"Input\n{input[0].tolist()}")

# print(vocab)

import yaml
from pathlib import Path

args_model_params = "../data/tinystories_config.yaml"
args_checkpoint_file = "../data/tinystories_checkpoint.pt"

checkpoint_path = Path(args_checkpoint_file)
model_params = None
with open(args_model_params,"r") as f_in:
    yaml_data = yaml.safe_load(f_in)
    model_params = yaml_data['model_params']

if model_params is None:
    logger.error(f"Could not load model config file {args_model_params}")
    sys.exit(1)

batch_size = model_params['batch_size']
context_length = model_params['context_length']
embedding_dim = model_params['embedding_dim']
num_layers = model_params['num_layers']
num_heads = model_params['num_heads']
feed_forward_dim = model_params['feed_forward_dim']
rope_theta = model_params['rope_theta']
end_of_text = model_params['end_of_text']
temperature = model_params['temperature']
max_tokens_generated = model_params['max_tokens_generated']
softmax_nucleus_threshold = model_params['softmax_nucleus_threshold']

vocab_size = len(vocab)

# create the language model
model:torch.nn.Module = custom_modules.TransformerLm(vocab_size=vocab_size,
                                              context_length=context_length,
                                              num_layers=num_layers,
                                              d_model=embedding_dim,
                                              num_heads=num_heads,
                                              d_ff=feed_forward_dim,
                                              rope_theta=rope_theta
                                              )


# create the optimizer
optimizer = custom_training.AdamW(params=model.parameters(),lr=1e-3,
                            weight_decay=0.01, betas=(0.9, 0.999), eps=1e-8)
softmax = custom_modules.Softmax(temperature=temperature)

tokens_generated = input[0].tolist()

if checkpoint_path.is_file():
    t = custom_training.run_load_checkpoint(src=checkpoint_path,model=model,optimizer=optimizer)
    logger.info(f"Loading checkpoint file {checkpoint_path} and starting at iteration {t}")

token_iter = 0
best_token = -1

def get_end_of_text_id(vocab:dict[int,bytes],end_of_text:str)->int:
    eot_bytes = end_of_text.encode('utf-8')
    for k,v in vocab.items():
        if v == eot_bytes:
            return k
    return -1

end_of_text_id = get_end_of_text_id(vocab=vocab,end_of_text=end_of_text)
assert(end_of_text_id>-1)

def mask_weights(weights:torch.Tensor, softmax_nucleus_threshold:float = 1.0):
    values,indices = torch.sort(weights,descending=True)
    # logger.debug(f"sorted {values.shape} indices {indices.shape}")
    last_index = weights.shape[-1]
    i = 0
    sum = 0
    # logger.debug(f"Weight shape is {weights.shape}")
    mask = torch.zeros(weights.shape,dtype=bool,requires_grad=False)
    while i < last_index:
        # logger.debug(f"values shape {values.shape}")
        sum+=values[i]
        # logger.debug(f"sum is {sum} at {i}")
        mask[indices[i]] = True
        if sum >= softmax_nucleus_threshold:
            break
        i+=1
    # print(f"mask {mask}")
    weight_masked = torch.where(mask,weights,0)
    return weight_masked

while True:
    logger.debug(f"At iteration {token_iter}")
    if best_token==end_of_text_id or token_iter==max_tokens_generated:
        break
    # logger.debug(f"input {input}")
    model_output = model(input)
    probs =  softmax(in_features = model_output,dim=1)
    last_pos_prob = probs[0,-1]
    probs = mask_weights(weights = last_pos_prob, softmax_nucleus_threshold= softmax_nucleus_threshold)

    # logger.debug(f"last_pos_prob {last_pos_prob.shape}\n{last_pos_prob}")
    last_index_sampled = torch.multinomial(last_pos_prob, num_samples=1, replacement=True)
    logger.debug(f"last index {last_index_sampled}")
    input = torch.cat((input[...,1:],last_index_sampled.view(1,-1)),dim=-1)
    logger.debug(f"input is now {input}")

    # probs =  softmax(in_features = model_output,dim=1)
    # best_indices =  torch.argmax(probs,dim=2)
    # logger.debug(f"best indices {best_indices}")
    # best_index = torch.unsqueeze(best_indices[...,-1],dim=0)
    # logger.debug(f"best {best_index}")
    # input = torch.cat((input[...,1:],best_index),dim=-1)


    best_token = last_index_sampled.item()
    tokens_generated.append(best_token)
    logger.debug(f"Tokens so far {tokens_generated}")
    token_iter+=1
    # break
    # append to our prompt and extended tokens

    # logger.debug(f"last_prob {last_prob.item()} {type(last_prob.item)}")
    # logger.debug(f"last_prob {last_prob} probs {probs.shape}\n{probs}")
    # logger.debug(f"input {input}")
    # input = torch.cat((input[...,1:],best_token),dim=-1)
    # tokens_generated.append(best_token)
    # logger.debug(f"added {added}")
    # maxcols = torch.argmax(probs,dim=-1)
    # logger.debug(f"maxcols are {maxcols.shape}\n{maxcols}")
    # logger.debug(f"Tokens generated {tokens_generated}")


output = tkn.decode(tokens_generated)
logger.info(f"Output is {output}")

cs336_basics.tokenizer - INFO - 2026-10-14 09:30:48,070 - Tokenizer created.
__main__ - DEBUG - 2026-10-14 09:30:48,084 - Input
[72, 101, 382]
__main__ - DEBUG - 2026-10-14 09:30:48,084 - Input
[72, 101, 382]
cs336_basics.custom_training - DEBUG - 2026-10-14 09:30:48,220 - AdamW created
__main__ - INFO - 2026-10-14 09:30:48,240 - Loading checkpoint file ../data/tinystories_checkpoint.pt and starting at iteration 999
__main__ - INFO - 2026-10-14 09:30:48,240 - Loading checkpoint file ../data/tinystories_checkpoint.pt and starting at iteration 999
__main__ - DEBUG - 2026-10-14 09:30:48,242 - At iteration 0
__main__ - DEBUG - 2026-10-14 09:30:48,242 - At iteration 0
__main__ - DEBUG - 2026-10-14 09:30:48,301 - last index tensor([492])
__main__ - DEBUG - 2026-10-14 09:30:48,301 - last index tensor([492])
__main__ - DEBUG - 2026-10-14 09:30:48,305 - input is now tensor([[101, 382, 492]])
__main__ - DEBUG - 2026-10-14 09:30:48,305 - input is now tensor([[101, 382, 492]])
__main__ - DEBUG - 2

In [ ]:


def mask_weights(weights:torch.Tensor, threshold:float = 1.0):
    values,indices = torch.sort(weights,descending=True)
    print(f"sorted {values} indices {indices}")
    last_index = weights.shape[-1]
    i = 0
    sum = 0
    mask = torch.zeros(weights.shape,dtype=bool)
    while i < last_index:
        sum+=values[0,i]
        # print(f"sum is {sum} at {i}")
        mask[0][indices[0][i]] = True
        if sum >= threshold:
            break
        i+=1
    # print(f"mask {mask}")
    weight_masked = np.where(mask,weights,0)
    return weight_masked

# Weights do not need to sum to 1, but must be non-negative
weights = torch.tensor([[0.1,0.2, 0.6, 0.1]])

weight_masked = mask_weights(weights,0.85)

print(f"masked weights {weight_masked}")

# Draw 4 samples with replacement
# indices = torch.multinomial(weights, num_samples=1, replacement=True)
# print("Multinomial Indices:", indices)
# Example output: tensor([1, 1, 0, 1])


In [66]:
a=[1,2,3,4]
a[-4]


1